# Customer Segmentation with K-Means

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from pathlib import Path
from urllib.parse import quote_plus
from sqlalchemy import create_engine

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

In [ ]:
server = os.getenv("SQL_SERVER", "localhost")
database = "EcommerceAnalysis"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

encoded_connection_string = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={encoded_connection_string}"
)

In [ ]:
sql_path = Path("..") / "01_data_preparation" / "01_customer_dataset.sql"

customer_query = sql_path.read_text(encoding="utf-8")

df = pd.read_sql(customer_query, engine)

## 2. Feature Preparation

In [ ]:
candidate_features = [
    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_items_per_order",
    "category_diversity",
    "favorite_category_share",
    "purchase_frequency",
    "effective_discount_rate",
    "returned_order_rate"
]

X = df[candidate_features].copy()

In [ ]:
log_features = [
    "frequency",
    "monetary",
    "avg_order_value",
    "purchase_frequency"
]

for feature in log_features:
    X[f"{feature}_log"] = np.log1p(X[feature])

In [ ]:
X_model = X[
    [
        "recency_days",
        "frequency_log",
        "monetary_log",
        "avg_order_value_log",
        "avg_items_per_order",
        "category_diversity",
        "favorite_category_share",
        "purchase_frequency_log",
        "effective_discount_rate",
        "returned_order_rate"
    ]
].copy()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_model)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=X_model.columns,
    index=X_model.index
)

## 3. K-Means Clustering

In [ ]:
k_values = range(2, 11)
inertias = []

In [ ]:
for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled_df)

    inertias.append(kmeans.inertia_)

### Elbow Method

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, inertias, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(k_values)
plt.show()

### Silhouette Score

In [ ]:
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_scaled_df)

    score = silhouette_score(
        X_scaled_df,
        cluster_labels,
        sample_size=5000,
        random_state=42
    )

    silhouette_scores.append(score)

In [ ]:
silhouette_scores

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, silhouette_scores, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for K-Means")

plt.xticks(k_values)
plt.show()

## 4. Feature Set Evaluation

Evaluate whether removing potentially redundant or less informative features improves the quality of the customer clusters.

In [ ]:
X_experiment_1 = X_scaled_df.drop(columns='category_diversity')

In [ ]:
silhouette_scores_exp1 = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_experiment_1)

    score = silhouette_score(
        X_experiment_1,
        cluster_labels,
        sample_size=5000,
        random_state=42
    )

    silhouette_scores_exp1.append(score)

### Experiment 1 – Removing Category Diversity

`category_diversity` was removed because of its strong correlation with `frequency`.

The removal did not produce a meaningful overall improvement in Silhouette Score. Although the score for K=2 increased slightly, scores for several other K values decreased.

**Decision:** Retain `category_diversity` in the clustering feature set.

In [ ]:
X_experiment_2 = X_scaled_df.drop(columns='returned_order_rate')

In [ ]:
silhouette_scores_exp2 = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_experiment_2)

    score = silhouette_score(
        X_experiment_2,
        cluster_labels,
        sample_size=5000,
        random_state=42
    )

    silhouette_scores_exp2.append(score)

### Experiment 2 – Removing Returned Order Rate

`returned_order_rate` was removed to evaluate whether its highly zero-heavy distribution was limiting cluster separation.

Removing the feature improved the Silhouette Score for several K values, particularly K=2, although the improvement was not consistent across all cluster counts.

**Decision:** Exclude `returned_order_rate` from the primary clustering feature set and retain it for post-clustering customer profiling.

In [ ]:
X_experiment_3 = X_scaled_df.drop(columns="favorite_category_share")

In [ ]:
silhouette_scores_exp3 = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_experiment_3)

    score = silhouette_score(
        X_experiment_3,
        cluster_labels,
        sample_size=5000,
        random_state=42
    )

    silhouette_scores_exp3.append(score)

### Experiment 3 – Removing Favorite Category Share

`favorite_category_share` was removed to evaluate its overlap with `category_diversity`.

The removal reduced cluster separation for K=2, while producing moderate improvements for several higher K values. The effect was therefore not consistently positive.

**Decision:** Retain `favorite_category_share` in the primary clustering feature set.

In [ ]:
X_final = X_scaled_df.drop(columns="returned_order_rate")

## 5. Clustering Model Evaluation

Based on the feature set experiments, `returned_order_rate` was excluded from the primary clustering inputs, while the remaining behavioral features were retained.

In [ ]:
k_values = range(2, 11)
final_inertias = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_final)

    final_inertias.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, final_inertias, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method – Final Feature Set")

plt.xticks(k_values)
plt.show()

### Alternative Clustering Model

K-Means was used as the baseline clustering algorithm. However, the Elbow Method did not reveal a clear cluster structure, and the Silhouette Scores indicated relatively modest separation between customer groups.

Since customer behavior may not form clearly separated centroid-based clusters, Gaussian Mixture Model (GMM) was evaluated as an alternative clustering approach.

Unlike K-Means, which assigns each customer to the nearest cluster centroid, GMM models clusters probabilistically and allows more flexible cluster shapes and overlapping customer behavior patterns.

#### GMM Component Evaluation

In [ ]:
component_values = range(2, 11)

aic_scores = []
bic_scores = []

In [ ]:
for n in component_values:
    gmm = GaussianMixture(
        n_components=n,
        random_state=42
    )

    gmm.fit(X_final)

    aic_scores.append(gmm.aic(X_final))
    bic_scores.append(gmm.bic(X_final))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(component_values, aic_scores, marker="o", label="AIC")
plt.plot(component_values, bic_scores, marker="o", label="BIC")

plt.xlabel("Number of Components")
plt.ylabel("Score")
plt.title("AIC and BIC for Gaussian Mixture Model")
plt.xticks(component_values)
plt.legend()

plt.show()

Both AIC and BIC reached their minimum values at 9 components, indicating that the 9-component GMM provides the best balance between model fit and complexity among the evaluated models.

However, statistical model fit alone is not sufficient for customer segmentation. The resulting cluster sizes, separation, and behavioral interpretability should also be evaluated before selecting the final clustering model.

#### 9-Component GMM

In [ ]:
gmm_final = GaussianMixture(
    n_components=9,
    random_state=42
)

gmm_labels = gmm_final.fit_predict(X_final)

In [ ]:
pd.Series(gmm_labels).value_counts().sort_index()

In [ ]:
gmm_silhouette = silhouette_score(
    X_final,
    gmm_labels,
    sample_size=5000,
    random_state=42
)

gmm_silhouette

In [ ]:
gmm_silhouette_scores = []

for n in component_values:
    gmm = GaussianMixture(
        n_components=n,
        random_state=42
    )

    labels = gmm.fit_predict(X_final)

    score = silhouette_score(
        X_final,
        labels,
        sample_size=5000,
        random_state=42
    )

    gmm_silhouette_scores.append(score)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    silhouette_scores_exp2,
    marker="o",
    label="K-Means"
)

plt.plot(
    component_values,
    gmm_silhouette_scores,
    marker="o",
    label="GMM"
)

plt.xlabel("Number of Clusters / Components")
plt.ylabel("Silhouette Score")
plt.title("K-Means vs GMM – Silhouette Score")
plt.xticks(k_values)
plt.legend()

plt.show()

### Model Comparison

K-Means and Gaussian Mixture Model (GMM) were evaluated using the same final feature set.

GMM achieved its lowest AIC and BIC values with 9 components, suggesting that this configuration provided the best probabilistic fit among the evaluated GMM models. However, its Silhouette Score was approximately 0.007, indicating very weak separation between the resulting customer groups.

Across comparable cluster counts, K-Means consistently produced higher Silhouette Scores than GMM. This indicates that K-Means generated more clearly separated customer groups for the current feature space.

**Decision:** K-Means was selected as the primary clustering algorithm. GMM was retained as an alternative model evaluation, demonstrating that better probabilistic model fit does not necessarily result in better-separated customer segments.

## 6. Final Cluster Selection

Although K=2 achieved the highest Silhouette Score, statistical separation alone does not determine the most useful customer segmentation. Candidate solutions with K=2, K=3, and K=4 were therefore compared based on their behavioral profiles and business interpretability.

In [ ]:
profile_features = [
    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_items_per_order",
    "category_diversity",
    "favorite_category_share",
    "purchase_frequency",
    "effective_discount_rate",
    "returned_order_rate"
]

for k in [2, 3, 4]:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_final)

    profile = df[profile_features].copy()
    profile["cluster"] = labels

    print(f"\nK = {k}")
    display(
        profile.groupby("cluster")[profile_features]
        .mean()
        .round(2)
    )

### Final K Selection

K=2 achieved the highest Silhouette Score, but it produced two broad customer groups with limited behavioral granularity.

K=3 provided a more detailed separation between lower-, medium-, and higher-value customer behavior. K=4 revealed an additional distinct group characterized by very low recency and substantially higher purchase frequency, representing a meaningful behavioral pattern that was not clearly isolated in the simpler solutions.

Although K=4 has a lower Silhouette Score than K=2, it provides a more informative balance between cluster separation and behavioral interpretability.

**Decision:** K=4 was selected for the final K-Means segmentation.

## 7. Final Customer Segmentation

In [ ]:
final_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

final_cluster_labels = final_kmeans.fit_predict(X_final)

In [ ]:
clustered_df = df.copy()

clustered_df["cluster"] = final_cluster_labels

In [ ]:
clustered_df["cluster"].value_counts().sort_index()

### Cluster Profiling

The final clusters are analyzed using the original customer metrics to understand the behavioral characteristics of each customer segment.

In [ ]:
profile_features = [
    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_items_per_order",
    "category_diversity",
    "favorite_category_share",
    "purchase_frequency",
    "effective_discount_rate"
]

cluster_profile = (
    clustered_df
    .groupby("cluster")[profile_features]
    .mean()
    .round(2)
)

cluster_profile

In [ ]:
additional_profile_features = [
    "returned_order_rate",
    "average_rating",
    "review_rate"
]

additional_cluster_profile = (
    clustered_df
    .groupby("cluster")[additional_profile_features]
    .mean()
    .round(2)
)

additional_cluster_profile

### Segment Interpretation

The final clusters were assigned descriptive segment names based on their purchasing behavior, customer value, recency, and purchase frequency.

- **Cluster 0 – Low-Value Infrequent Customers:** Low spending and purchase frequency with relatively old purchase activity.
- **Cluster 1 – High-Value Less-Active Customers:** Highest historical spending and purchase frequency, but relatively low recent activity.
- **Cluster 2 – Active High-Value Customers:** High customer value combined with very recent and frequent purchasing activity.
- **Cluster 3 – Moderate-Value Less-Active Customers:** Moderate historical value but relatively low recent purchasing activity.

Return rates and average ratings showed limited differences across segments and were therefore not treated as primary segment-defining characteristics.

In [ ]:
segment_mapping = {
    0: "Low-Value Infrequent Customers",
    1: "High-Value Less-Active Customers",
    2: "Active High-Value Customers",
    3: "Moderate-Value Less-Active Customers"
}

clustered_df["segment_name"] = clustered_df["cluster"].map(segment_mapping)

In [ ]:
segment_distribution = (
    clustered_df["segment_name"]
    .value_counts()
    .rename_axis("segment_name")
    .reset_index(name="customer_count")
)

segment_distribution["percentage"] = (
    segment_distribution["customer_count"]
    / len(clustered_df)
    * 100
).round(2)

segment_distribution

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    segment_distribution["segment_name"],
    segment_distribution["customer_count"]
)

plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.title("Customer Segment Distribution")
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()

## 8. Summary

Customer segmentation was performed using behavioral and value-based features derived from historical purchase activity.

K-Means was initially evaluated using the Elbow Method and Silhouette Score. Feature set experiments showed that excluding `returned_order_rate` improved cluster separation, while `category_diversity` and `favorite_category_share` were retained.

Gaussian Mixture Model (GMM) was also evaluated as an alternative clustering approach. Although AIC and BIC favored a 9-component GMM, its clusters showed weak separation based on Silhouette Score. K-Means produced more clearly separated customer groups and was therefore selected as the final clustering algorithm.

Although K=2 achieved the highest K-Means Silhouette Score, K=4 provided a more informative behavioral segmentation by identifying a distinct group of recently active, high-value customers.

The final segmentation identified four customer groups:

- **Low-Value Infrequent Customers:** 4,360 customers (19.96%)
- **High-Value Less-Active Customers:** 5,928 customers (27.13%)
- **Active High-Value Customers:** 2,345 customers (10.73%)
- **Moderate-Value Less-Active Customers:** 9,216 customers (42.18%)

The resulting customer-level segment labels can be used for downstream customer analysis and business reporting.